# Week 1: Foundations & Spark Architecture

This notebook walks through the same core ideas as `main.py` cell by cell:
- Creating a local `SparkSession`
- Seeing lazy evaluation on transformations (`filter`, `withColumn`, `select`)
- Inspecting the execution plan with `.explain()`
- Triggering actions (`show`, `count`) and checking the Spark UI at `http://localhost:4040`
- Stopping the `SparkSession` cleanly at the end

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = (
    SparkSession.builder.appName("Week1-Spark-Architecture")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print(f"Spark Version: {spark.version}")
print("Spark UI available at: http://localhost:4040")

## Lazy Transformations vs. Actions

Transformations like `filter()`, `withColumn()`, and `select()` do not compute results right away. Spark records them in a logical/physical plan until an **action** asks for output.

In [ ]:
sample_rows = [
    ("evt-1", "US", 120.0),
    ("evt-2", "IN", 45.0),
    ("evt-3", "CA", 650.0),
    ("evt-4", "US", 15.0),
    ("evt-5", "DE", 980.0),
]

df = spark.createDataFrame(sample_rows, ["event_id", "country", "amount"])

# Lazy transformations: recorded in the plan, not executed yet
transformed_df = (
    df.filter(col("amount") >= 50.0)
    .withColumn(
        "order_size",
        when(col("amount") >= 500.0, "large").otherwise("medium"),
    )
    .select("event_id", "country", "amount", "order_size")
)

transformed_df.explain(mode="simple")

## Trigger Actions & Inspect the Spark UI

Running `show()` and `count()` triggers actual Spark jobs. While `SparkSession` is still running, you can open `http://localhost:4040` in your browser to inspect the jobs and stages.

In [ ]:
transformed_df.show(truncate=False)
print(f"Filtered row count: {transformed_df.count()}")

## Clean Shutdown

Always stop the `SparkSession` when you are finished exploring so local Spark processes release their resources cleanly.

In [ ]:
spark.stop()
print("SparkSession stopped cleanly.")